In [ ]:
import visualizer
visualizer.start_server()


  EVOLANGDIFF CONTINUOUS MANIFOLD OBSERVER RUNNING
  Open your browser at: http://localhost:8008



127.0.0.1 - - [09/Nov/2026 12:31:13] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:13] "GET /api/state HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:14] "POST /api/probe HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:15] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:15] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:15] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:15] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:15] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:16] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:16] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:16] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:16] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:16] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:16] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [09/Nov/2026 12:31:16] "POST

In [1]:
"""
test_phonetics.py — Definitive Test Suite for Autonomous Continuous Phonetics.
Validates:
  1. Homophony & Lexical Ambiguity Friction
  2. Intra-word Modulation & Anti-Stutter (OCP)
  3. Continuous Evolutionary Vowel Dispersion (Escaping the Schwa Trap)
  4. Dynamic Hiatus Resolution & Clustering
  5. Metrical Syncope & Pulmonary Erosion
"""

from __future__ import annotations

import math
import random
import numpy as np
from phonetics import (
    Phoneme,
    transition_cost,
    repair_phonemes,
    word_energy,
    relax_gesture_gradient,
    apply_syncope_and_clustering,
    is_vowel_gesture,
    render_word_trajectory,
)
from word import Word
from semantics import SemanticVector, DerivationRecord


def make_dummy_word(ipa_phonemes: list[Phoneme], sem_coords: list[float], word_id: int = 1) -> Word:
    return Word(
        word_id=word_id,
        vector=SemanticVector(*sem_coords),
        phonemes=ipa_phonemes,
        derivation=DerivationRecord(root_family_id=word_id),
    )


def test_homophony_ambiguity_friction():
    print("\n--- Test 1: Homophony & Ambiguity Friction ---")
    # Base word: /taku/ ("water")
    p_t = Phoneme([1.0, 0.0, 0.0, 0.25, 0.80, 0.50, 0.0, 0.60])
    p_a = Phoneme([1.0, 0.0, 1.0, 0.0, 0.95, 0.40, 0.0, 0.40])
    p_k = Phoneme([1.0, 0.0, 1.0, 0.0, 0.00, 0.75, 0.0, 0.60])
    p_u = Phoneme([0.30, 1.0, 1.0, 0.0, 0.28, 0.92, 0.0, 0.40])

    w_water = make_dummy_word([p_t, p_a, p_k, p_u], [1.0, 0.0, 0.7, 0.5, 0.5, 0.1, 0.4], word_id=101)

    # Isolated energy cost
    e_isolated = word_energy(w_water, cohort_sample=[])

    # Case A: Cohort has distinct sounding word /pime/ ("kill")
    p_p = Phoneme([0.0, 0.0, 1.0, 0.0, 0.80, 0.50, 0.0, 0.60])
    p_i = Phoneme([1.0, 0.0, 1.0, 0.0, 0.28, 0.05, 0.0, 0.40])
    p_m = Phoneme([0.0, 0.0, 1.0, 0.0, 0.80, 0.50, 0.60, 0.40])
    p_e = Phoneme([1.0, 0.0, 1.0, 0.0, 0.52, 0.10, 0.0, 0.40])
    w_kill = make_dummy_word([p_p, p_i, p_m, p_e], [0.6, 0.8, 0.1, 0.95, 0.95, 0.1, 0.4], word_id=102)

    e_distinct = word_energy(w_water, cohort_sample=[w_water, w_kill])

    # Case B: Cohort has homophonic collision (another concept sounding identical to /taku/)
    w_homophone = make_dummy_word([p_t, p_a, p_k, p_u], [0.6, 0.8, 0.1, 0.95, 0.95, 0.1, 0.4], word_id=103)
    e_colliding = word_energy(w_water, cohort_sample=[w_water, w_homophone])

    print(f"Isolated Energy:    {e_isolated:.3f} J")
    print(f"Distinct Cohort:    {e_distinct:.3f} J (diff: +{e_distinct - e_isolated:.3f} J)")
    print(f"Homophonic Clash:   {e_colliding:.3f} J (diff: +{e_colliding - e_isolated:.3f} J)")

    assert e_colliding > e_distinct > e_isolated, "Ambiguity potential must heavily tax homophonic collisions!"
    print("✓ Homophony ambiguity friction correctly penalizes conflicting homophones.")


def test_intra_word_modulation_and_ocp():
    print("\n--- Test 2: Intra-Word Modulation & OCP Anti-Stutter ---")
    # Slushy stutter word: /ɐəɐmɐvɐ/
    p_near_open = Phoneme([1.0, 0.0, 1.0, 0.0, 0.85, 0.50, 0.0, 0.40])
    p_schwa = Phoneme([1.0, 0.0, 1.0, 0.0, 0.60, 0.50, 0.0, 0.40])
    p_m = Phoneme([0.0, 0.0, 1.0, 0.0, 0.80, 0.50, 0.60, 0.40])
    p_v = Phoneme([0.08, 0.0, 1.0, 0.0, 0.80, 0.50, 0.0, 0.40])

    w_slush = make_dummy_word([p_near_open, p_schwa, p_near_open, p_m, p_near_open, p_v, p_near_open], [0.5] * 7, 201)

    # Crisp alternating word: /takumi/
    p_t = Phoneme([1.0, 0.0, 0.0, 0.25, 0.80, 0.50, 0.0, 0.60])
    p_a = Phoneme([1.0, 0.0, 1.0, 0.0, 0.95, 0.40, 0.0, 0.40])
    p_k = Phoneme([1.0, 0.0, 1.0, 0.0, 0.00, 0.75, 0.0, 0.60])
    p_u = Phoneme([0.30, 1.0, 1.0, 0.0, 0.28, 0.92, 0.0, 0.40])
    p_i = Phoneme([1.0, 0.0, 1.0, 0.0, 0.28, 0.05, 0.0, 0.40])

    w_crisp = make_dummy_word([p_t, p_a, p_k, p_u, p_m, p_i], [0.5] * 7, 202)

    e_slush = word_energy(w_slush)
    e_crisp = word_energy(w_crisp)

    # OCP check on transition cost between /ɐ/ and /ə/ vs /t/ and /a/
    cost_adjacent_vowels = transition_cost(p_near_open, p_schwa)
    cost_consonant_vowel = transition_cost(p_t, p_a)

    print(f"Slushy Word ({w_slush.form}) Energy:  {e_slush:.3f} J")
    print(f"Crisp Word  ({w_crisp.form}) Energy:  {e_crisp:.3f} J")
    print(f"Transition Cost [ɐ -> ə] (OCP Stutter): {cost_adjacent_vowels:.3f} J")
    print(f"Transition Cost [t -> a] (Jaw Cycle):   {cost_consonant_vowel:.3f} J")

    assert cost_adjacent_vowels > cost_consonant_vowel, "Adjacent near-identical vowels must incur OCP penalty!"
    assert e_slush > e_crisp, "Slushy words with low modulation must cost more energy than crisp CV words!"
    print("✓ Intra-word modulation and OCP properly penalize phonological stagnation.")


def test_vowel_dispersion():
    print("\n--- Test 3: Vowel Dispersion & Escape from the Schwa Trap ---")
    rng = random.Random(42)
    # Start a vowel right in the center (schwa: Height=0.60, Backness=0.50)
    p_center = Phoneme([1.0, 0.0, 1.0, 0.0, 0.60, 0.50, 0.0, 0.40])
    print(f"Initial State: /{p_center.ipa}/ (H={p_center.g[4]:.2f}, B={p_center.g[5]:.2f})")

    # Simulate 80 drift iterations under Hamiltonian gradient descent
    p_curr = p_center
    history = [p_curr.ipa]

    # Neighboring pool with another vowel pressing from below
    neighbor_pool = [np.array([1.0, 0.0, 1.0, 0.0, 0.70, 0.50, 0.0, 0.40], dtype=np.float32)]

    for _ in range(80):
        p_curr = relax_gesture_gradient(
            p=p_curr,
            adjacent_p=None,
            neighbor_g_pool=neighbor_pool,
            valence_intensity=0.8,
            bias=np.zeros(8, dtype=np.float32),
            alt=0.0, hum=0.5, temp=0.5, t_drift=1.5,
            rng=rng
        )
        history.append(p_curr.ipa)

    print(f"Final State:   /{p_curr.ipa}/ (H={p_curr.g[4]:.2f}, B={p_curr.g[5]:.2f})")
    assert p_curr.ipa not in {'ə', 'ɐ'}, f"Vowel failed to escape central trap, got /{p_curr.ipa}/!"
    print(f"✓ Vowel successfully dispersed to peripheral vowel: /{p_curr.ipa}/.")


def test_hiatus_resolution():
    print("\n--- Test 4: Dynamic Hiatus Resolution & Coalescence ---")
    # Sequence of two identical central vowels in hiatus: /ə/ + /ə/
    p_schwa1 = Phoneme([1.0, 0.0, 1.0, 0.0, 0.60, 0.50, 0.0, 0.40], duration=0.8)
    p_schwa2 = Phoneme([1.0, 0.0, 1.0, 0.0, 0.60, 0.50, 0.0, 0.40], duration=0.8)
    repaired_same = repair_phonemes([p_schwa1, p_schwa2])

    print(f"Hiatus [ə + ə] -> {[p.ipa for p in repaired_same]} (len={len(repaired_same)})")
    assert len(repaired_same) == 1, "Identical adjacent vowels must coalesce!"
    assert repaired_same[0].duration > 1.0, "Coalesced vowel must be lengthened!"

    # Sequence of two contrastive vowels: /i/ + /a/
    p_i = Phoneme([1.0, 0.0, 1.0, 0.0, 0.28, 0.05, 0.0, 0.40], duration=1.0)
    p_a = Phoneme([1.0, 0.0, 1.0, 0.0, 0.95, 0.40, 0.0, 0.40], duration=1.0)
    repaired_diff = repair_phonemes([p_i, p_a])
    diff_ipas = [p.ipa for p in repaired_diff]

    print(f"Hiatus [i + a] -> {diff_ipas} (len={len(repaired_diff)})")
    assert len(repaired_diff) == 3, "Contrastive hiatus must insert an organic glide to break vowel hiatus!"
    print("✓ Hiatus repair coalesces near-identical vowels and inserts glides between contrastive vowels.")


def test_erosion_and_syncope():
    print("\n--- Test 5: Pulmonary Syncope & Articulatory Erosion ---")
    # 4-syllable word in alpine arid environment: /tatatata/
    p_t = Phoneme([1.0, 0.0, 0.0, 0.25, 0.80, 0.50, 0.0, 0.60])
    p_a = Phoneme([1.0, 0.0, 1.0, 0.0, 0.95, 0.40, 0.0, 0.40], duration=0.9)
    chain = [p_t, p_a, p_t, p_a, p_t, p_a, p_t, p_a]

    # Extreme altitude and aridity
    eroded = apply_syncope_and_clustering(chain, usage_frequency=3.0, alt=0.85, hum=0.15, temp=0.20)
    orig_form, _ = render_word_trajectory(np.stack([p.vector for p in chain]))
    eroded_form, _ = render_word_trajectory(np.stack([p.vector for p in eroded]))

    print(f"Before Erosion: /{orig_form}/ ({len(chain)} segments)")
    print(f"After Syncope:  /{eroded_form}/ ({len(eroded)} segments)")

    assert len(eroded) < len(chain), "Unstressed vowels in harsh pulmonary environments must erode!"
    print("✓ Syncope properly condenses speech trajectories under environmental load.")


if __name__ == "__main__":
    print("==================================================================")
    print("  RUNNING EVOLANGDIFF CONTINUOUS PHONETICS TEST SUITE")
    print("==================================================================")
    test_homophony_ambiguity_friction()
    test_intra_word_modulation_and_ocp()
    test_vowel_dispersion()
    test_hiatus_resolution()
    test_erosion_and_syncope()
    print("\n==================================================================")
    print("  ALL TESTS PASSED. Autonomous physical balance verified.")
    print("==================================================================")

  RUNNING EVOLANGDIFF CONTINUOUS PHONETICS TEST SUITE

--- Test 1: Homophony & Ambiguity Friction ---
Isolated Energy:    3.128 J
Distinct Cohort:    3.181 J (diff: +0.053 J)
Homophonic Clash:   6.258 J (diff: +3.130 J)
✓ Homophony ambiguity friction correctly penalizes conflicting homophones.

--- Test 2: Intra-Word Modulation & OCP Anti-Stutter ---
Slushy Word (ɐˈəɐmɐvɐ) Energy:  5.635 J
Crisp Word  (ˈtakumi) Energy:  4.876 J
Transition Cost [ɐ -> ə] (OCP Stutter): 1.460 J
Transition Cost [t -> a] (Jaw Cycle):   0.329 J
✓ Intra-word modulation and OCP properly penalize phonological stagnation.

--- Test 3: Vowel Dispersion & Escape from the Schwa Trap ---
Initial State: /ə/ (H=0.60, B=0.50)
Final State:   /ɲ/ (H=0.00, B=0.01)
✓ Vowel successfully dispersed to peripheral vowel: /ɲ/.

--- Test 4: Dynamic Hiatus Resolution & Coalescence ---
Hiatus [ə + ə] -> ['ə'] (len=1)
Hiatus [i + a] -> ['i', 'j', 'a'] (len=3)
✓ Hiatus repair coalesces near-identical vowels and inserts glides between